# Local recommender training notebook

This notebook runs the repository's local batch workflow on an already prepared encoded dataset. It fits the two-tower model, evaluates it against the popularity baseline on validation data, applies the serving-quality gate, and exports a checked model bundle plus run manifest.

The training settings below match the batch pipeline defaults. Outputs go to a new timestamped directory under artifacts. The test split is not used for model selection. Run this notebook from a Python 3.11 + Java 21 Linux/WSL kernel as documented in the README; Spark on the configured Windows Python 3.14 interpreter is not supported for this workflow.

This is local-only: the notebook does not create AWS resources or deploy the API.

In [1]:
from pathlib import Path
import os
import platform
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from the repository checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project:", ROOT)
print("Python:", sys.version.split()[0], platform.platform())
print("Interpreter:", sys.executable)

if sys.platform != "linux" or sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "Select the project's Linux/WSL Python 3.11 Jupyter kernel before training. "
        "This repository's Spark workflow does not run on the configured Windows Python 3.14 kernel."
    )

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("recommender-local-training-notebook")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

Project: /mnt/c/Users/aksha/PycharmProjects/EndtoEndMLProject
Python: 3.11.13 Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.43
Interpreter: /home/akshay/miniconda3/bin/python


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/akshay/miniconda3/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/16 11:37:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.2.0


In [2]:
from datetime import datetime, timezone
import json

DATA_DIR = ROOT / "data" / "processed" / "movies_tv_5m" / "two_tower" / "v1"
required_paths = [DATA_DIR / name for name in ("train", "validation", "item_mapping")]
missing = [path for path in required_paths if not path.is_dir()]
if missing:
    raise FileNotFoundError(
        "Prepared dataset folders are missing: "
        + ", ".join(str(path) for path in missing)
        + ". Prepare the data using the README instructions first."
    )

MAX_USERS = 2_000
MAX_TRAIN_PAIRS = 100_000
EPOCHS = 10
SEED = 42
K_VALUES = (10, 20)
SHUFFLE_PARTITIONS = 32
EXPERIMENT_NAME = "two-tower-recommender"

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = ROOT / "artifacts" / f"local_training_{run_stamp}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
MLFLOW_URI = f"sqlite:///{OUTPUT_DIR / 'mlflow.db'}"

print("Encoded data:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)
print("Run bounds:", {"max_users": MAX_USERS, "max_train_pairs": MAX_TRAIN_PAIRS, "epochs": EPOCHS, "seed": SEED})
print("Tracking store:", MLFLOW_URI)

Encoded data: /mnt/c/Users/aksha/PycharmProjects/EndtoEndMLProject/data/processed/movies_tv_5m/two_tower/v1
Output directory: /mnt/c/Users/aksha/PycharmProjects/EndtoEndMLProject/artifacts/local_training_20261016T183743Z
Run bounds: {'max_users': 2000, 'max_train_pairs': 100000, 'epochs': 10, 'seed': 42}
Tracking store: sqlite:////mnt/c/Users/aksha/PycharmProjects/EndtoEndMLProject/artifacts/local_training_20261016T183743Z/mlflow.db


In [3]:
# Sanity-check split presence and basic schemas before fitting.
from pyspark.sql import functions as F

split_counts = {}
for split_name in ("train", "validation"):
    frame = spark.read.parquet(str(DATA_DIR / split_name))
    expected_columns = {"user_idx", "item_idx"}
    if not expected_columns.issubset(frame.columns):
        raise ValueError(f"{split_name} must contain columns {sorted(expected_columns)}")
    split_counts[split_name] = frame.count()

item_mapping = spark.read.parquet(str(DATA_DIR / "item_mapping"))
if "item_idx" not in item_mapping.columns:
    raise ValueError("item_mapping must contain item_idx")
item_count = item_mapping.select("item_idx").distinct().count()
if item_count < 2:
    raise ValueError(f"Need at least two mapped items, found {item_count}")

print({"split_rows": split_counts, "catalog_items": item_count})
display(item_mapping.orderBy("item_idx").limit(5))

{'split_rows': {'train': 2687407, 'validation': 237857}, 'catalog_items': 137070}


DataFrame[item_id: string, item_idx: bigint]

In [4]:
from dataclasses import asdict
from src.models.train_two_tower import (
    TrainingConfig,
    _read_sampled_pairs,
    train_two_tower,
)
from src.models.experiment_tracking import log_experiment_run

train_pairs, validation_pairs, model_user_ids, num_items = _read_sampled_pairs(
    spark,
    DATA_DIR,
    MAX_USERS,
    MAX_TRAIN_PAIRS,
    SEED,
)
print({
    "sampled_users": len(model_user_ids),
    "train_pairs": len(train_pairs),
    "validation_pairs": len(validation_pairs),
    "catalog_items": num_items,
})

training_config = TrainingConfig(epochs=EPOCHS, seed=SEED, device="cpu")
checkpoint_path = OUTPUT_DIR / "best_model.pt"
training_result = train_two_tower(
    train_pairs,
    validation_pairs,
    num_users=len(model_user_ids),
    num_items=num_items,
    config=training_config,
    model_user_ids=model_user_ids,
    item_mapping_path=str(DATA_DIR / "item_mapping"),
    checkpoint_path=checkpoint_path,
)

history_path = OUTPUT_DIR / "history.json"
history_path.write_text(
    json.dumps({
        "best_epoch": training_result.best_epoch,
        "best_validation_loss": training_result.best_validation_loss,
        "train_pairs": len(train_pairs),
        "validation_pairs": len(validation_pairs),
        "model_users": len(model_user_ids),
        "catalog_items": num_items,
        "epochs": training_result.history,
    }, indent=2),
    encoding="utf-8",
)
training_run_id = log_experiment_run(
    experiment_name=EXPERIMENT_NAME,
    run_name=f"train-seed-{SEED}-users-{len(model_user_ids)}",
    parameters={
        **asdict(training_config),
        "max_users": MAX_USERS,
        "max_train_pairs": MAX_TRAIN_PAIRS,
        "actual_users": len(model_user_ids),
        "actual_train_pairs": len(train_pairs),
        "actual_validation_pairs": len(validation_pairs),
        "catalog_items": num_items,
        "data_path": str(DATA_DIR),
    },
    metrics={
        "best_validation_bpr_loss": training_result.best_validation_loss,
        "best_epoch": float(training_result.best_epoch),
    },
    step_metrics=training_result.history,
    artifacts=[history_path, checkpoint_path],
    tags={"stage": "training", "model": "two_tower"},
    tracking_uri=MLFLOW_URI,
)
(OUTPUT_DIR / "training_mlflow_run.json").write_text(
    json.dumps({
        "run_id": training_run_id,
        "experiment_name": EXPERIMENT_NAME,
        "tracking_uri": MLFLOW_URI,
    }, indent=2),
    encoding="utf-8",
)

print(f"Best epoch: {training_result.best_epoch}")
print(f"Best validation BPR loss: {training_result.best_validation_loss:.6f}")
print("Checkpoint:", checkpoint_path)

{'sampled_users': 2000, 'train_pairs': 22313, 'validation_pairs': 2000, 'catalog_items': 137070}
epoch=1 train_loss=0.693144 validation_loss=0.693142
epoch=2 train_loss=0.692550 validation_loss=0.693142
epoch=3 train_loss=0.690663 validation_loss=0.693101
epoch=4 train_loss=0.685705 validation_loss=0.692989
epoch=5 train_loss=0.676198 validation_loss=0.692733
epoch=6 train_loss=0.661573 validation_loss=0.692302
epoch=7 train_loss=0.641827 validation_loss=0.691679
epoch=8 train_loss=0.618091 validation_loss=0.690902
epoch=9 train_loss=0.590582 validation_loss=0.689953
epoch=10 train_loss=0.561049 validation_loss=0.688930


2026/10/16 11:38:07 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/16 11:38:07 INFO mlflow.store.db.utils: Updating database tables
2026/10/16 11:38:10 INFO mlflow.tracking.fluent: Experiment with name 'two-tower-recommender' does not exist. Creating a new experiment.


Best epoch: 10
Best validation BPR loss: 0.688930
Checkpoint: /mnt/c/Users/aksha/PycharmProjects/EndtoEndMLProject/artifacts/local_training_20261016T183743Z/best_model.pt


In [5]:
from src.models.evaluate_two_tower import evaluate_validation

# Compute the fallback while this notebook's Spark session is still active.
# evaluate_validation manages and stops its own Spark session.
popularity_rows = (
    spark.read.parquet(str(DATA_DIR / "train"))
    .groupBy("item_idx")
    .count()
    .orderBy("count", "item_idx", ascending=[False, True])
    .select("item_idx")
    .collect()
)
popularity_indices = [int(row.item_idx) for row in popularity_rows]

validation_report = evaluate_validation(
    checkpoint_path=checkpoint_path,
    data_path=DATA_DIR,
    ks=K_VALUES,
    shuffle_partitions=SHUFFLE_PARTITIONS,
    split="validation",
)
validation_path = OUTPUT_DIR / "validation_metrics.json"
validation_path.write_text(json.dumps(validation_report, indent=2), encoding="utf-8")

logged_metrics = {}
for method, metrics_by_k in validation_report["metrics_at_k"].items():
    for k, values in metrics_by_k.items():
        for metric_name in ("recall", "precision", "ndcg"):
            logged_metrics[f"{method}.{metric_name}_at_{k}"] = float(values[metric_name])
for cohort, cohort_result in validation_report["metrics_by_cohort"].items():
    for method, metrics_by_k in cohort_result["metrics_at_k"].items():
        for k, values in metrics_by_k.items():
            for metric_name in ("recall", "precision", "ndcg"):
                logged_metrics[f"{cohort}.{method}.{metric_name}_at_{k}"] = float(values[metric_name])

evaluation_run_id = log_experiment_run(
    experiment_name=EXPERIMENT_NAME,
    run_name=f"evaluate-validation-{OUTPUT_DIR.name}",
    parameters={
        "checkpoint": str(checkpoint_path),
        "data_path": str(DATA_DIR),
        "users_evaluated": validation_report["users_evaluated"],
        "catalog_items": validation_report["catalog_items"],
        "k_values": ",".join(map(str, K_VALUES)),
    },
    metrics=logged_metrics,
    artifacts=[validation_path],
    tags={"stage": "validation_evaluation", "model": "two_tower"},
    tracking_uri=MLFLOW_URI,
)
(OUTPUT_DIR / "evaluation_mlflow_run.json").write_text(
    json.dumps({
        "run_id": evaluation_run_id,
        "experiment_name": EXPERIMENT_NAME,
        "tracking_uri": MLFLOW_URI,
        "split": "validation",
    }, indent=2),
    encoding="utf-8",
)

display(validation_report["metrics_at_k"])

26/10/16 11:38:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


{'two_tower': {'10': {'users': 2000,
   'recall': 0.009,
   'precision': 0.0009000000000000002,
   'ndcg': 0.004225384733037108},
  '20': {'users': 2000,
   'recall': 0.015,
   'precision': 0.0007500000000000003,
   'ndcg': 0.005747135201599962}},
 'popularity_baseline_on_same_user_sample': {'10': {'users': 2000,
   'recall': 0.0155,
   'precision': 0.0015500000000000008,
   'ndcg': 0.007535297806938515},
  '20': {'users': 2000,
   'recall': 0.0225,
   'precision': 0.001125,
   'ndcg': 0.009349440962125002}}}

In [6]:
from datetime import datetime, timezone
import hashlib
import uuid
from src.models.register_model import check_validation_gate
from src.pipeline.batch_job import export_serving_artifacts

try:
    validation_gate = {"passed": True, **check_validation_gate(validation_report)}
except ValueError as error:
    validation_gate = {"passed": False, "reason": str(error)}

serving_policy = "two_tower" if validation_gate["passed"] else "popularity"
(OUTPUT_DIR / "validation_gate.json").write_text(
    json.dumps(validation_gate, indent=2), encoding="utf-8"
)

# popularity_indices was collected in the evaluation cell before its managed
# Spark session stopped the notebook's shared SparkContext.
pipeline_run_id = str(uuid.uuid4())
model_version = "sha256:" + hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
lineage = {
    "model_version": model_version,
    "pipeline_run_id": pipeline_run_id,
    "training_mlflow_run_id": training_run_id,
    "evaluation_mlflow_run_id": evaluation_run_id,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "serving_policy": serving_policy,
}
bundle_path = export_serving_artifacts(
    DATA_DIR,
    OUTPUT_DIR / "serving",
    checkpoint_path,
    lineage=lineage,
    popularity_item_indices=popularity_indices,
)

batch_manifest = {
    **lineage,
    "input_data": str(DATA_DIR),
    "users": validation_report["users_evaluated"],
    "validation_metrics": validation_report["metrics_at_k"],
    "validation_gate": validation_gate,
    "serving_policy": serving_policy,
    "validation_report": str(validation_path),
    "serving_bundle": str(bundle_path),
}
manifest_path = OUTPUT_DIR / "batch_manifest.json"
manifest_path.write_text(json.dumps(batch_manifest, indent=2), encoding="utf-8")
if not bundle_path.is_file() or bundle_path.stat().st_size == 0:
    raise RuntimeError("The serving bundle was not exported correctly")

print("Validation gate:", validation_gate)
print("Selected serving policy:", serving_policy)
print("Serving bundle:", bundle_path, f"({bundle_path.stat().st_size:,} bytes)")
print("Run manifest:", manifest_path)

Validation gate: {'passed': False, 'reason': 'validation gate failed: NDCG@10 lift -43.925% is below 5.000%'}
Selected serving policy: popularity
Serving bundle: /mnt/c/Users/aksha/PycharmProjects/EndtoEndMLProject/artifacts/local_training_20261016T183743Z/model_bundle.zip (17,488,210 bytes)
Run manifest: /mnt/c/Users/aksha/PycharmProjects/EndtoEndMLProject/artifacts/local_training_20261016T183743Z/batch_manifest.json
